In [1]:



# Cell 2: Import all necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_nlp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

print("✅ Libraries imported successfully.")

# Cell 3: Load the data
try:
    train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")
    test_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/test.csv")
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("🛑 Data files not found. Make sure the competition data is added to your notebook.")

2025-09-09 06:38:13.218003: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757399893.389058      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757399893.440183      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Libraries imported successfully.
✅ Data loaded successfully.


In [2]:
# Cell 4: Prepare Text and Labels

# 1. Combine text fields into a single input string
train_df['input_text'] = train_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)
test_df['input_text'] = test_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)

# 2. Prepare labels for multi-label classification
# Ensure columns are strings to prevent errors
train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']

# Use MultiLabelBinarizer to create multi-hot encoded vectors
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df['full_label'].apply(lambda x: [x]))

print("✅ Text and labels prepared.")

✅ Text and labels prepared.


In [3]:
# Cell 5: Create a Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train_df['input_text'], 
    y_train, 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Training samples: 29356
Validation samples: 7340


In [4]:
MODEL_PATH = "/kaggle/input/deberta_v3/keras/deberta_v3_base_en/3"

# Calculate the number of unique labels
num_labels = len(mlb.classes_)

# Load the entire classifier in one go. 
# It automatically includes the preprocessor and a classification head.
classifier = keras_nlp.models.DebertaV3Classifier.from_preset(
    "deberta_v3_base_en",
    num_classes=num_labels,
)

# Compile the model. By using "binary_crossentropy" as the loss, Keras
# will automatically use a 'sigmoid' activation, which is correct for multi-label tasks.
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)

classifier.summary()
print("✅ DebertaV3Classifier loaded and compiled successfully.")


I0000 00:00:1757399908.348413      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Preprocessor: "deberta_v3_text_classifier_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ deberta_v3_tokenizer (DebertaV3Tokenizer)                     │                      Vocab size: 128,001 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "deberta_v3_text_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ deberta_v3_backbone           │ (None, None, 768)         │     183,831,552 │ padding_mask[0][0],        │
│ (DebertaV3Backbone)           │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item (GetItem)            │ (None, 768)               │               0 │ deberta_v3_backbone[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pooled_dropout (Dropout)      │ (None, 768)               │               0 │ get_item[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pooled_dense (Dense)          │ (None, 768)               │         590,592 │ pooled_dropout[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 768)               │               0 │ pooled_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 65)                │          49,985 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 184,472,129 (703.71 MB)

 Trainable params: 184,472,129 (703.71 MB)

 Non-trainable params: 0 (0.00 B)

✅ DebertaV3Classifier loaded and compiled successfully.


In [5]:
# Cell 4: Manually Create TensorFlow Datasets
BATCH_SIZE = 8

# Create the training dataset from our pandas/numpy objects
tf_train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
# Shuffle, batch, and prefetch for optimal performance
tf_train_dataset = tf_train_dataset.shuffle(buffer_size=len(X_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create the validation dataset
tf_val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
tf_val_dataset = tf_val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Data converted to the tf.data.Dataset format.")

✅ Data converted to the tf.data.Dataset format.


In [6]:
print("⏳ Starting model training...")

# The .fit() command starts the training process.
# - epochs: How many times the model will see the entire training dataset.
# - batch_size: How many samples the model works on at once.
# Cell 5: Train the Model
history = classifier.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=3
)

print("✅ Model training complete.")

print("✅ Model training complete.")

⏳ Starting model training...
Epoch 1/3


I0000 00:00:1757399987.980913      60 service.cc:148] XLA service 0x790d2c02b740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757399987.981559      60 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1757399995.112349      60 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1757400030.855326      60 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2514s 654ms/step - binary_accuracy: 0.9841 - loss: 0.0881 - val_binary_accuracy: 0.9868 - val_loss: 0.0826
Epoch 2/3
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2346s 639ms/step - binary_accuracy: 0.9858 - loss: 0.0849 - val_binary_accuracy: 0.9879 - val_loss: 0.0907
Epoch 3/3
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 2345s 639ms/step - binary_accuracy: 0.9837 - loss: 0.0885 - val_binary_accuracy: 0.9846 - val_loss: 0.0933
✅ Model training complete.
✅ Model training complete.


In [7]:
# Cell: Prediction and Submission

print("⏳ Generating predictions on the test set...")

# 1. Prepare the test data's input text column
test_df['input_text'] = test_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)

# 2. Get the predicted probabilities from the trained classifier
# The classifier automatically preprocesses the raw text.
test_probabilities = classifier.predict(test_df['input_text'])

# 3. Find the indices of the top 3 probabilities for each row
top_3_indices = np.argsort(test_probabilities, axis=1)[:, ::-1][:, :3]

# 4. Use the MultiLabelBinarizer (mlb) to convert indices back to label names
top_3_labels = mlb.classes_[top_3_indices]

# 5. Join the 3 labels with a space to match the submission format
predictions_str = [" ".join(labels) for labels in top_3_labels]

# 6. Create the final submission DataFrame
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'], 
    'Category:Misconception': predictions_str
})

# 7. Save the DataFrame to a csv file
submission_df.to_csv('submission.csv', index=False)

print("✅ submission.csv file created successfully!")
print("Here's a preview:")
print(submission_df.head())

⏳ Generating predictions on the test set...
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
✅ submission.csv file created successfully!
Here's a preview:
   row_id                             Category:Misconception
0   36696  False_Misconception:Incomplete True_Correct:na...
1   36697  False_Misconception:Incomplete True_Correct:na...
2   36698  False_Misconception:Incomplete True_Correct:na...
